# MATCHA-Bench: reproduce the experiments on a local GPU

This notebook reruns the full pipeline:
1. **Pre-training (DAPT)** of `hfl/chinese-bert-wwm-ext` on MATCHA with three masking strategies:
   random, the depression lexicon (Chinese MentalBERT), and the 520-word multi-disorder lexicon.
2. **Fine-tuning** of all models on 5 public ACL tasks and 7 MATCHA tasks, with 3 seeds each.
3. **TF-IDF baseline, results tables and significance tests.**

### What you need
- An NVIDIA GPU with CUDA and PyTorch (≥ 12 GB VRAM recommended; smaller GPUs: see `SMALL_GPU`).
  Internet access is used to download the base models from Hugging Face.
- The code: `git clone https://github.com/shehryars715/matcha-bench.git` (or let the next cell clone it).
- The data zip **`matcha-bench-data.zip`** (DAPT corpus, tasks, lexicons; 137 MB), received from the
  authors. It is derived from MATCHA and covered by its Data Use Agreement, so do not redistribute it.

### Time
Roughly 30–40 GPU-hours on one T4-class GPU, less on faster GPUs. The run is **resumable**: if it is
interrupted, run the notebook again. Finished steps are skipped and pre-training continues from its
last checkpoint. Progress is logged to `outputs/pipeline_log.jsonl`.

In [ ]:
from pathlib import Path

MODE = "check"            # "check" = 5-minute sanity test, "full" = the real pipeline
DATA_ZIP = Path("matcha-bench-data.zip")      # path to the data zip you received
REPO = Path("matcha-bench")                   # where the code is (cloned if missing)
SMALL_GPU = False         # True for GPUs under ~12 GB: halves batch sizes (same effective batch)
INSTALL_REQUIREMENTS = True

# If this notebook is opened from inside the cloned repo (notebooks/), use that repo.
if (Path.cwd().parent / "scripts" / "run_pipeline.py").exists():
    REPO = Path.cwd().parent
REPO = REPO.resolve()
print("repo:", REPO)

In [ ]:
import os, subprocess, sys

def sh(*args, cwd=None):
    """Run a command and stream its output into the notebook."""
    print("$", " ".join(map(str, args)), flush=True)
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd or REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
                            env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

if not (REPO / "scripts").exists():
    sh("git", "clone", "https://github.com/shehryars715/matcha-bench.git", REPO, cwd=REPO.parent)
sh("git", "log", "-1", "--format=code version: %h %s")
if INSTALL_REQUIREMENTS:
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

import torch
assert torch.cuda.is_available(), "No CUDA GPU visible to PyTorch"
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# Unpack the data into the repo (data/dapt, data/tasks, lexicons/*.txt).
import zipfile
if not (REPO / "data" / "dapt" / "train.npz").exists():
    assert DATA_ZIP.exists(), f"Data zip not found: {DATA_ZIP.resolve()}"
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(REPO)
    print("data unpacked")
for p in ["data/dapt/train.npz", "data/tasks/suicide/meta.json", "lexicons/lexicon_multidisorder.txt"]:
    print(("ok      " if (REPO / p).exists() else "MISSING ") + p)

## Run

In [ ]:
small = (["--dapt-set", "training.per_device_train_batch_size=16",
          "--finetune-set", "defaults.batch_size=8", "defaults.grad_accum=2"] if SMALL_GPU else [])

if MODE == "check":
    # Tiny randomly initialised BERT: checks data, masking, training and fine-tuning end to end.
    sh(sys.executable, "scripts/run_dapt.py", "--config", "configs/dapt/smoke.yaml")
    sh(sys.executable, "scripts/run_finetune.py", "--tasks", "suicide", "--models", "outputs/dapt/smoke/final",
       "--seeds", "42", "--set", "defaults.epochs=1", "output_root=outputs/check_finetune")
    print("\nCheck passed. Set MODE = 'full' to run the real pipeline.")
else:
    sh(sys.executable, "scripts/run_pipeline.py", "--no-limit", *small)

## Results
Mean ± std over 3 seeds, in percent (`f1` is each task's primary metric: macro, binary or micro F1).
Paired-bootstrap tests compare each lexicon-guided model with random masking.

In [ ]:
import glob
import pandas as pd
log = REPO / "outputs" / "pipeline_log.jsonl"
if log.exists():
    print(log.read_text(encoding="utf-8")[-3000:])
for name in ("summary_f1.csv", "summary_macro_f1.csv"):
    path = REPO / "outputs" / "results" / name
    if path.exists():
        print(name); display(pd.read_csv(path, index_col=0))
for path in sorted(glob.glob(str(REPO / "outputs" / "results" / "significance_*.csv"))):
    print(Path(path).name); display(pd.read_csv(path).groupby("task")[["delta", "p_value"]].mean())